# FIFA World Cup Prediction Model
## Notebook 06 — Results Comparison

**Run this after the World Cup ends (July 19, 2026)**

Compare our pre-tournament predictions against what actually happened. This is the ultimate validation of the model.

### What we measure
| Metric | Description |
|---|---|
| Brier score | How close our win probabilities were to actual outcomes |
| Log loss | Penalises confident wrong predictions heavily |
| Champion accuracy | Did we predict the right winner? |
| Top N accuracy | Was the actual winner in our top N? |
| Stage accuracy | Did we correctly predict how far each team went? |


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (13, 6)})

os.makedirs(os.path.join('..', 'outputs'), exist_ok=True)
print('Imports OK ✓')


## 1. Load pre-tournament predictions

In [ ]:
predictions = pd.read_csv(os.path.join('..', 'outputs', 'wc2026_predictions.csv'))
print(f'Loaded predictions for {len(predictions)} teams')
print()
print('Top 10 predicted:')
print(predictions[['team', 'win_pct', 'final_pct', 'semifinal_pct']].head(10).to_string(index=False))


## 2. Enter actual tournament results

Fill in the actual results below after the World Cup ends on July 19, 2026.
Set each team's furthest stage reached.


In [ ]:
# ── FILL IN AFTER THE WORLD CUP ENDS ──
# Stage options: 'group_exit', 'r32_exit', 'r16_exit', 
#                'quarterfinal_exit', 'semifinal_exit', 'final_exit', 'winner'

actual_results = {
    # Group A
    "Mexico":                  "r32_exit",       # ← update with real result
    "South Africa":            "group_exit",
    "South Korea":             "group_exit",
    "Czechia":                 "group_exit",

    # Group B
    "Canada":                  "r32_exit",
    "Bosnia and Herzegovina":  "group_exit",
    "Qatar":                   "group_exit",
    "Switzerland":             "r16_exit",

    # Group C
    "Brazil":                  "quarterfinal_exit",
    "Morocco":                 "r16_exit",
    "Haiti":                   "group_exit",
    "Scotland":                "group_exit",

    # Group D
    "United States":           "r32_exit",
    "Paraguay":                "group_exit",
    "Australia":               "group_exit",
    "Turkey":                  "r16_exit",

    # Group E
    "Germany":                 "semifinal_exit",
    "Curaçao":                 "group_exit",
    "Ivory Coast":             "group_exit",
    "Ecuador":                 "r32_exit",

    # Group F
    "Netherlands":             "quarterfinal_exit",
    "Japan":                   "r16_exit",
    "Sweden":                  "group_exit",
    "Tunisia":                 "group_exit",

    # Group G
    "Belgium":                 "r16_exit",
    "Egypt":                   "group_exit",
    "Iran":                    "group_exit",
    "New Zealand":             "group_exit",

    # Group H
    "Spain":                   "winner",          # ← update with real result
    "Cape Verde":              "group_exit",
    "Saudi Arabia":            "group_exit",
    "Uruguay":                 "r32_exit",

    # Group I
    "France":                  "final_exit",
    "Senegal":                 "r32_exit",
    "Iraq":                    "group_exit",
    "Norway":                  "r16_exit",

    # Group J
    "Argentina":               "semifinal_exit",
    "Algeria":                 "group_exit",
    "Austria":                 "r32_exit",
    "Jordan":                  "group_exit",

    # Group K
    "Portugal":                "r16_exit",
    "DR Congo":                "group_exit",
    "Uzbekistan":              "group_exit",
    "Colombia":                "r32_exit",

    # Group L
    "England":                 "quarterfinal_exit",
    "Croatia":                 "r32_exit",
    "Ghana":                   "group_exit",
    "Panama":                  "group_exit",
}

# Stage ranking (higher = further)
STAGE_RANK = {
    'group_exit':         0,
    'r32_exit':           1,
    'r16_exit':           2,
    'quarterfinal_exit':  3,
    'semifinal_exit':     4,
    'final_exit':         5,
    'winner':             6,
}

actual_winner = [t for t, s in actual_results.items() if s == 'winner']
print(f'Actual winner: {actual_winner[0] if actual_winner else "Not set yet"}')
print(f'Teams entered: {len(actual_results)}')


## 3. Build comparison DataFrame

In [ ]:
# Map actual stage to numeric rank
actual_df = pd.DataFrame([
    {"team": team, "actual_stage": stage, "actual_rank": STAGE_RANK[stage]}
    for team, stage in actual_results.items()
])

# Merge with predictions
comparison = predictions.merge(actual_df, on='team', how='inner')

# Predicted stage rank (using win_pct thresholds)
def pred_stage(row):
    if row['win_pct'] > 10:       return 6  # predicted winner
    elif row['final_pct'] > 18:   return 5  # predicted finalist
    elif row['semifinal_pct'] > 12: return 4
    elif row['quarterfinal_pct'] > 20: return 3
    elif row['r16_pct'] > 40:     return 2
    elif row['r32_pct'] > 60:     return 1
    else:                          return 0

comparison['predicted_rank'] = comparison.apply(pred_stage, axis=1)
comparison['rank_error'] = abs(comparison['predicted_rank'] - comparison['actual_rank'])
comparison = comparison.sort_values('actual_rank', ascending=False).reset_index(drop=True)

print(comparison[['team', 'win_pct', 'actual_stage', 'predicted_rank', 'actual_rank', 'rank_error']].to_string(index=False))


## 4. Model evaluation metrics

In [ ]:
# Champion prediction
predicted_winner = predictions.iloc[0]['team']
actual_win = actual_winner[0] if actual_winner else None
predicted_rank_of_winner = predictions[predictions.team == actual_win].index[0] + 1 if actual_win else None

print('=== Champion Prediction ===')
print(f'  Predicted winner:       {predicted_winner} ({predictions.iloc[0]["win_pct"]:.1f}%)')
print(f'  Actual winner:          {actual_win}')
print(f'  Actual winner was ranked #{predicted_rank_of_winner} in our predictions')
print(f'  Correct: {predicted_winner == actual_win}')
print()

# Top N accuracy
print('=== Top N Accuracy ===')
for n in [1, 3, 5, 8, 10]:
    top_n_teams = set(predictions.head(n)['team'])
    correct = actual_win in top_n_teams if actual_win else False
    print(f'  Winner in top {n:2d}: {correct}')
print()

# Mean absolute error on stage rank
mae = comparison['rank_error'].mean()
print(f'=== Stage Prediction MAE ===')
print(f'  Mean absolute stage error: {mae:.2f} rounds')
print(f'  (0 = perfect, 1 = off by one round, etc.)')
print()

# Brier score for win probability
actual_won = (comparison['actual_stage'] == 'winner').astype(int)
brier = ((comparison['win_pct'] / 100 - actual_won) ** 2).mean()
print(f'=== Brier Score (win probability) ===')
print(f'  {brier:.4f}  (lower is better, 0 = perfect, 0.25 = random)')


## 5. Visualise predicted vs actual

In [ ]:
# Predicted win% vs actual stage for all teams
fig, ax = plt.subplots(figsize=(14, 7))

stage_labels = {0:'Group', 1:'R32', 2:'R16', 3:'QF', 4:'SF', 5:'Final', 6:'Winner'}
stage_colors = {0:'#374151', 1:'#6b7280', 2:'#3b82f6', 3:'#8b5cf6', 4:'#f59e0b', 5:'#ef4444', 6:'#FFD700'}

top20 = comparison.head(20)
colors = [stage_colors[r] for r in top20['actual_rank']]

bars = ax.bar(range(len(top20)), top20['win_pct'], color=colors, edgecolor='white', linewidth=0.5)

ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20['team'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Predicted championship probability (%)')
ax.set_title('Predicted Win % vs Actual Tournament Finish — Top 20', fontweight='bold', fontsize=13)

# Legend
patches = [mpatches.Patch(color=stage_colors[k], label=stage_labels[k]) for k in sorted(stage_colors.keys())]
ax.legend(handles=patches, title='Actual stage', loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'prediction_vs_actual.png'), bbox_inches='tight')
plt.show()


In [ ]:
# Calibration: did higher predicted probabilities correspond to better actual performance?
fig, ax = plt.subplots(figsize=(10, 6))

for stage, rank in STAGE_RANK.items():
    subset = comparison[comparison['actual_rank'] == rank]
    if len(subset) > 0:
        ax.scatter(
            subset['win_pct'],
            [rank] * len(subset),
            label=stage_labels[rank],
            color=stage_colors[rank],
            s=80, alpha=0.8, zorder=3
        )
        for _, row in subset.iterrows():
            ax.annotate(row['team'], (row['win_pct'], rank),
                       textcoords='offset points', xytext=(4, 0), fontsize=7, alpha=0.8)

ax.set_xlabel('Predicted championship probability (%)')
ax.set_ylabel('Actual stage reached')
ax.set_yticks(list(stage_labels.keys()))
ax.set_yticklabels(list(stage_labels.values()))
ax.set_title('Model calibration — predicted probability vs actual stage', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'calibration_actual.png'), bbox_inches='tight')
plt.show()


In [ ]:
# Biggest surprises — teams we got most wrong
comparison['surprise_score'] = comparison.apply(
    lambda r: (r['actual_rank'] - r['predicted_rank']) * (r['win_pct'] / 10 + 1),
    axis=1
)

overachievers = comparison.nlargest(5, 'surprise_score')[['team', 'win_pct', 'actual_stage', 'surprise_score']]
underachievers = comparison.nsmallest(5, 'surprise_score')[['team', 'win_pct', 'actual_stage', 'surprise_score']]

print('=== Biggest Overachievers (did better than predicted) ===')
print(overachievers.to_string(index=False))
print()
print('=== Biggest Underachievers (did worse than predicted) ===')
print(underachievers.to_string(index=False))


## 6. Save summary report

In [ ]:
# Save full comparison to CSV
out_path = os.path.join('..', 'outputs', 'wc2026_results_comparison.csv')
comparison.to_csv(out_path, index=False)
print(f'Saved → {out_path}')

# Print final summary
print()
print('=' * 50)
print('       FINAL MODEL REPORT CARD')
print('=' * 50)
print(f'  Champion correctly predicted: {predicted_winner == actual_win}')
print(f'  Actual winner ranked:         #{predicted_rank_of_winner}')
print(f'  Mean stage error:             {mae:.2f} rounds')
print(f'  Brier score:                  {brier:.4f}')
print(f'  Total teams tracked:          {len(comparison)}')
print('=' * 50)


## ✅ Analysis complete

Update the `actual_results` dictionary in cell 2 with the real outcomes after July 19, 2026, then run all cells to get the full model evaluation.

Don't forget to push the outputs to GitHub:
```bash
git add outputs/wc2026_results_comparison.csv outputs/prediction_vs_actual.png outputs/calibration_actual.png
git commit -m "results: WC2026 prediction vs actual comparison"
git push
```
